# Исследование рекомендательных систем и машинного обучения

Разделы анализа:
1. Сводка по данным и расчет разреженности матрицы (Sparsity);
2. Распределение оценок и распределение фильмов по жанрам;
3. Сравнение качества моделей (RMSE, MAE, Precision, Recall);
4. 2D PCA проекция признаков фильмов;
5. Матрица сходства фильмов (Cosine Similarity Heatmap).

In [ ]:
import sys
import os
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer

from database.connection import SessionLocal
from database.models import Movie, Rating, User
from ml_engine.evaluation import MLEvaluationEngine

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline
print('Библиотеки успешно загружены')

## 1. Сводка по данным и расчет разреженности матрицы

In [ ]:
db = SessionLocal()
eval_engine = MLEvaluationEngine()
summary = eval_engine.get_dataset_summary(db)

print('Сводка по датасету:')
for k, v in summary.items():
    print(f'  {k}: {v}')

## 2. Распределение оценок и фильмов по жанрам

In [ ]:
ratings = db.query(Rating).all()
df_ratings = pd.DataFrame([{'user_id': r.user_id, 'movie_id': r.movie_id, 'rating': float(r.rating)} for r in ratings])

genre_data = eval_engine.get_genre_distribution(db)
df_genres = pd.DataFrame(genre_data).head(10)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(df_ratings['rating'], bins=9, kde=True, color='#4A90E2', edgecolor='black', ax=ax1)
ax1.set_title('Распределение пользовательских оценок (1.0 - 5.0)', fontsize=12)
ax1.set_xlabel('Оценка')
ax1.set_ylabel('Количество')
ax1.axvline(df_ratings['rating'].mean(), color='red', linestyle='--', label=f'Среднее: {df_ratings["rating"].mean():.2f}')
ax1.legend()

sns.barplot(data=df_genres, x='genre', y='count', color='#5DADE2', edgecolor='black', ax=ax2)
ax2.set_title('Фильмы по жанрам (Топ-10)', fontsize=12)
ax2.set_xlabel('Жанр')
ax2.set_ylabel('Количество')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3. Сравнение качества моделей: RMSE, MAE, Precision, Recall

In [ ]:
metrics_res = eval_engine.evaluate_models(db)
df_metrics = pd.DataFrame(metrics_res['models']).T
display(df_metrics)

fig, ax = plt.subplots(figsize=(8, 4))
df_metrics[['rmse', 'mae']].plot(kind='bar', color=['#E74C3C', '#F39C12'], edgecolor='black', ax=ax)
ax.set_title('Сравнение ошибок моделей (RMSE vs MAE)', fontsize=12)
ax.set_ylabel('Ошибка')
ax.set_xlabel('Модель')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

## 4. 2D PCA проекция признаков фильмов

In [ ]:
pca_data = eval_engine.get_pca_movie_projection(db)
df_pca = pd.DataFrame(pca_data)

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_pca, x='pc1', y='pc2', hue='genre', s=120, palette='tab10')

for _, row in df_pca.iterrows():
    plt.annotate(row['title'], (row['pc1']+0.01, row['pc2']+0.01), fontsize=8, alpha=0.85)

plt.title('2D PCA проекция признаков фильмов', fontsize=12)
plt.xlabel('Главная компонента 1 (PC1)')
plt.ylabel('Главная компонента 2 (PC2)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 5. Матрица сходства фильмов (Cosine Similarity Heatmap)

In [ ]:
sim_sample = eval_engine.get_similarity_heatmap_sample(db, limit=8)
titles = sim_sample['titles']
matrix = np.array(sim_sample['matrix'])
short_titles = [t[:15] + '..' if len(t) > 15 else t for t in titles]

plt.figure(figsize=(8, 6))
sns.heatmap(matrix, annot=True, fmt='.2f', cmap='Blues', xticklabels=short_titles, yticklabels=short_titles)
plt.title('Матрица сходства фильмов', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

db.close()